In [ ]:
# Cell 1 — Install
!pip install -q transformers accelerate flask pyngrok torch sentencepiece

In [ ]:
# Cell 2 — HuggingFace auth (need access to meta-llama/Meta-Llama-3-8B)
from huggingface_hub import login
login("Your_huggingface_token")   # or use Kaggle secrets

# Cell 3 — ngrok token for Instance 2 (must be a separate ngrok account)
import os
os.environ["NGROK_TOKEN"] = "Your_ngrok_token3" #microsoft-edge-avinash

In [ ]:
"""
╔══════════════════════════════════════════════════════════╗
║         LLAMA 3 8B — PIPELINE STAGE 2                   ║
║         Kaggle Instance 2                                ║
║         Layers: 22–31 + RMSNorm + LM Head               ║
║         Role: Recv hidden states → logits → token id    ║
╚══════════════════════════════════════════════════════════╝

SETUP (run these cells first in Kaggle):
  !pip install -q transformers accelerate flask pyngrok torch safetensors huggingface_hub
  !huggingface-cli login  # paste your HF token when prompted

IMPORTANT: Boot THIS notebook FIRST (before Stage 1 and Stage 0).
  Copy the ngrok URL it prints → paste into stage1_server.py as STAGE2_URL.
"""

# ── Imports ──────────────────────────────────────────────────────────────────
import os, io, time, copy
import torch
import numpy as np
from flask import Flask, request, jsonify
from transformers import AutoConfig, DynamicCache
from transformers.models.llama.modeling_llama import LlamaDecoderLayer, LlamaRMSNorm
import torch.nn as nn
from pyngrok import ngrok

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID     = "meta-llama/Meta-Llama-3-8B-Instruct"
SPLIT_LAYER  = 22          # Stage 2 owns layers 22..31 (inclusive)
TOTAL_LAYERS = 32
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE        = torch.float16

print(f"[Stage 2] Device: {DEVICE}  |  dtype: {DTYPE}")

# ── Model Loading ─────────────────────────────────────────────────────────────
class LlamaStage2(nn.Module):
    def __init__(self, config, split_layer=SPLIT_LAYER, total_layers=TOTAL_LAYERS):
        super().__init__()
        # ✅ Set attn implementation before building layers
        config._attn_implementation = "eager"
        self.config = config
        self.layers = nn.ModuleList([
            LlamaDecoderLayer(config, layer_idx=i)
            for i in range(split_layer, total_layers)
        ])
        self.norm    = LlamaRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding
        # ✅ Pass config directly — do NOT override hidden_size
        self.rotary_emb = LlamaRotaryEmbedding(config=config)

    def forward(self, hidden_states, past_key_values=None, past_len_override=0):
        seq_len  = hidden_states.shape[1]
        past_len = past_len_override
    
        position_ids   = torch.arange(past_len, past_len + seq_len,
                                      device=hidden_states.device).unsqueeze(0)
        cache_position = torch.arange(past_len, past_len + seq_len,
                                      device=hidden_states.device)
    
        if past_key_values is None:
            past_key_values = DynamicCache()
    
        cos, sin = self.rotary_emb(hidden_states, position_ids)
        position_embeddings = (cos, sin)
    
        hidden = hidden_states
        for layer in self.layers:
            out = layer(
                hidden,
                attention_mask=None,
                position_ids=position_ids,
                past_key_values=past_key_values,
                use_cache=True,              # ✅ must cache for autoregressive decode
                cache_position=cache_position,
                position_embeddings=position_embeddings,
            )
            hidden = out if isinstance(out, torch.Tensor) else out[0]
            if hidden.dim() == 2:
                hidden = hidden.unsqueeze(1)
    
        # ✅ Only return last position's hidden state to send downstream
        hidden = self.norm(hidden)
        logits = self.lm_head(hidden)
        
        return logits[:, -1:, :], past_key_values

def load_stage2_model():
    print("[Stage 2] Loading config...")
    config = AutoConfig.from_pretrained(MODEL_ID,
    trust_remote_code=True)

    print(f"[Stage 2] Building model (layers {SPLIT_LAYER}–{TOTAL_LAYERS - 1} + norm + lm_head)...")
    model = LlamaStage2(config)

    print(f"[Stage 2] Loading pretrained weights — targeted load, "
          f"layers {SPLIT_LAYER}–{TOTAL_LAYERS - 1} + norm + lm_head only...")
    # ── Targeted shard loader ─────────────────────────────────────────────────
    # Instead of loading the full 16 GB model into CPU RAM and then discarding
    # most of it, we read only the weight keys that belong to our layers directly
    # from the safetensors shards. This cuts CPU RAM usage from ~16 GB → ~5 GB
    # and reduces download time proportionally.
    from huggingface_hub import snapshot_download
    from safetensors import safe_open

    # Download only the safetensors shards (no PyTorch bin files needed)
    model_dir = snapshot_download(
        MODEL_ID,
        ignore_patterns=["*.bin", "*.bin.index.json"],   # skip old-format weights
    )

    # Build the set of parameter name prefixes we actually need
    # HF stores them as "model.layers.{i}....", "model.norm.", and "lm_head."
    our_prefixes = tuple(
        f"model.layers.{i}." for i in range(SPLIT_LAYER, TOTAL_LAYERS)
    ) + ("model.norm.", "lm_head.")

    # Walk every shard and pull out only our keys
    import glob, os as _os
    shard_files = sorted(glob.glob(_os.path.join(model_dir, "*.safetensors")))
    state_dict  = {}
    for shard_path in shard_files:
        with safe_open(shard_path, framework="pt", device="cpu") as f:
            for key in f.keys():
                if key.startswith(our_prefixes):
                    state_dict[key] = f.get_tensor(key).to(DTYPE)

    # Re-key to match our module's parameter names:
    #   "model.layers.{i}.X" → "layers.{local_i}.X"
    #   "model.norm.X"       → "norm.X"
    #   "lm_head.X"          → "lm_head.X"   (already correct)
    remapped = {}
    for key, tensor in state_dict.items():
        if key.startswith("model.layers."):
            parts      = key.split(".")            # ["model", "layers", "22", ...]
            global_idx = int(parts[2])
            local_idx  = global_idx - SPLIT_LAYER
            new_key    = "layers." + str(local_idx) + "." + ".".join(parts[3:])
        elif key.startswith("model.norm."):
            new_key = key[len("model."):]          # "norm.weight"
        else:
            new_key = key                          # "lm_head.weight"
        remapped[new_key] = tensor

    missing, unexpected = model.load_state_dict(remapped, strict=True)
    if missing:
        print(f"[Stage 2] ⚠️  Missing keys: {missing[:5]}{'...' if len(missing)>5 else ''}")
    # ─────────────────────────────────────────────────────────────────────────

    import gc; gc.collect()

    model = model.to(DTYPE).to(DEVICE).eval()
    print(f"[Stage 2] Model loaded. VRAM used: "
          f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")
    return model, config


# ── Sampling helpers ──────────────────────────────────────────────────────────
def top_p_sample(logits: torch.Tensor, temperature: float, top_p: float) -> int:
    """
    logits: (vocab_size,)
    Returns: sampled token id (int)
    """
    if temperature <= 0:
        return logits.argmax().item()

    logits = logits / temperature

    # Softmax
    probs = torch.softmax(logits, dim=-1)

    # Top-p (nucleus) filtering
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=0)
    # Remove tokens with cumulative prob above top_p (shift right to keep at least 1)
    sorted_probs[cumulative - sorted_probs > top_p] = 0.0
    sorted_probs /= sorted_probs.sum()

    # Sample
    sampled = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_idx[sampled].item()


# ── KV-Cache Store ────────────────────────────────────────────────────────────
# We keep a simple in-memory dict: session_id -> {"pkv_2": [...]}
# For research/single-session use this is fine.
SESSIONS = {}

# ── Flask App ─────────────────────────────────────────────────────────────────
app = Flask(__name__)

@app.route("/health")
def health():
    return jsonify({"stage": 2, "status": "ok",
                    "vram_gb": torch.cuda.memory_allocated() / 1e9})

@app.route("/forward", methods=["POST"])
def forward():
    session_id  = request.headers.get("X-Session-Id", "default")
    step        = int(request.headers.get("X-Step", 0))
    temperature = float(request.headers.get("X-Temperature", 0.8))
    top_p       = float(request.headers.get("X-Top-P", 0.95))
    past_len    = int(request.headers.get("X-Past-Len", 0))

    raw    = request.data
    hidden = torch.from_numpy(np.load(io.BytesIO(raw))).to(DTYPE).to(DEVICE)

    pkv_2 = SESSIONS.get(session_id, {}).get("pkv_2", None)

    with torch.no_grad():
        logits, pkv_2 = model(hidden, past_key_values=pkv_2,
                              past_len_override=past_len)

    SESSIONS[session_id] = {"pkv_2": pkv_2}

    logits_last = logits[0, -1, :]
    token_id    = top_p_sample(logits_last, temperature, top_p)
    logit_max   = float(logits_last.max().item())
    return jsonify({"token_id": token_id, "logit_max": logit_max})


@app.route("/reset_session", methods=["POST"])
def reset_session():
    """Clear KV cache for a session (call before a new conversation)."""
    session_id = request.json.get("session_id", "default")
    SESSIONS.pop(session_id, None)
    return jsonify({"cleared": session_id})


# ── Boot Sequence ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Load model
    model, config = load_stage2_model()

    # 2. Start ngrok — expose port 5002
    ngrok_token = os.environ.get("NGROK_TOKEN", "")   # set via Kaggle secrets
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
    tunnel = ngrok.connect(5002)

    print(f"\n{'='*60}")
    print(f"[Stage 2] PUBLIC URL: {tunnel.public_url}")
    print(f"  → Copy this URL into stage1_server.py as STAGE2_URL")
    print(f"  → e.g.  STAGE2_URL = \"{tunnel.public_url}\"")
    print(f"{'='*60}\n")

    # 3. Run Flask
    app.run(host="0.0.0.0", port=5002, threaded=False)